# Clase 116 — Regularización: L1/L2, dropout, max-norm, MC dropout

Técnicas para combatir el overfitting: penalizaciones **L1/L2**, **Dropout**, **MaxNorm**, **MC Dropout** (incertidumbre) y **Stochastic Depth** (redes residuales profundas).

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `matplotlib`.

### 🧠 Intuición previa

**Regularizar = penalizar la complejidad para que la red no memorice.**

Una red con muchos parámetros puede *aprender de memoria* el set de entrenamiento (incluido su ruido): baja el train loss a casi cero pero falla en datos nuevos. Regularizar es añadir una presión en contra de esa complejidad para forzar una solución más simple, que generalice:

- **L1/L2** suman una penalización al tamaño de los pesos (`|w|` o `w²`): pesos grandes = función retorcida = memoria; empujarlos hacia 0 la suaviza.
- **Dropout** apaga neuronas al azar en cada batch: la red no puede depender de una neurona concreta, así que reparte la representación (menos co-adaptación).
- **Max-norm** pone un techo a la norma de los pesos entrantes de cada neurona.

La idea común: **memorizar es fácil, generalizar es el objetivo**; la regularización hace que memorizar 'cueste' y por eso el modelo prefiere el patrón general antes que el detalle irrepetible.

## 1. Penalizaciones L1 / L2 / L1_L2

`l2` (weight decay) mantiene pesos chicos; `l1` promueve sparsity. Se pasan como `kernel_regularizer`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

reg_l2 = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dense(10,  activation="softmax"),
])
reg_l2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("l1:", keras.regularizers.l1(1e-4),
      "| l1_l2:", keras.regularizers.l1_l2(l1=1e-5, l2=1e-4))

## 2. Dropout

Enmascara una fracción `r` de activaciones por batch en training; en inference se desactiva (Keras escala automáticamente).

In [ ]:
con_dropout = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation="relu", kernel_initializer="he_normal"),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu", kernel_initializer="he_normal"),
    layers.Dropout(0.3),
    layers.Dense(10,  activation="softmax"),
])
con_dropout.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Dropout(0.3) entre capas; activo solo en training")

## 3. Max-norm constraint

Reescala `||w|| ≤ c` por neurona tras cada update (`kernel_constraint`).

In [ ]:
restringido = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(256, activation="relu", kernel_initializer="he_normal",
                 kernel_constraint=keras.constraints.MaxNorm(max_value=3.0)),
    layers.Dense(10, activation="softmax"),
])
print("MaxNorm(3.0) limita la norma de los pesos por unidad tras cada actualización")

## 4. Monte Carlo Dropout: incertidumbre

Con `training=True` en inference, N pasadas dan predicciones distintas → media ± std estiman la incertidumbre (Gal & Ghahramani 2016).

In [ ]:
x = tf.constant(np.random.default_rng(0).normal(size=(1, 784)), dtype=tf.float32)
predicciones = np.stack([con_dropout(x, training=True).numpy()[0] for _ in range(100)])
media = predicciones.mean(axis=0)
incertidumbre = predicciones.std(axis=0)
print("clase predicha:", int(media.argmax()))
print("std (incertidumbre) de la clase top:", round(float(incertidumbre[media.argmax()]), 4))

## 5. Stochastic Depth en un mini-ResNet

Dropear bloques residuales enteros al azar en training, con probabilidad **lineal** `p_i = i/N · p_max`.

In [ ]:
class StochasticDepth(layers.Layer):
    # Dropea el bloque residual entero con prob. drop_rate durante training.
    def __init__(self, drop_rate, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate

    def call(self, x, training=None):
        if not training or self.drop_rate == 0.0:
            return x
        keep = tf.cast(tf.random.uniform([]) >= self.drop_rate, x.dtype)
        return keep * x / (1.0 - self.drop_rate)          # inverted scaling

def bloque_residual(x, unidades, drop_rate):
    y = layers.Dense(unidades, activation="relu", kernel_initializer="he_normal")(x)
    y = layers.Dense(unidades, kernel_initializer="he_normal")(y)
    y = StochasticDepth(drop_rate)(y)
    return layers.Activation("relu")(layers.Add()([x, y]))

inp = keras.Input(shape=(128,))
x = layers.Dense(128)(inp)
n_bloques = 8
for i in range(n_bloques):
    x = bloque_residual(x, 128, drop_rate=i / n_bloques * 0.2)   # lineal 0 -> 0.2
salida = layers.Dense(10, activation="softmax")(x)
resnet = keras.Model(inp, salida)
print("mini-ResNet con Stochastic Depth lineal:", resnet.count_params(), "params")

## Ejercicios

1. **Baseline sin regularización**: entrená un MLP grande y observá el gap train/val (≥ 5 pp).
2. **L2 y Dropout**: agregá `l2(1e-3)` y luego `Dropout(0.3)`; compará el gap.
3. **MC Dropout**: 100 predicciones con `training=True` sobre imágenes ambiguas; reportá media ± std.
4. **Stochastic Depth**: mini-ResNet de 8 bloques con `p` lineal; compará contra sin stochastic depth.

## Conclusiones

- **L2** (weight decay) mantiene pesos chicos; **L1** promueve sparsity; `λ` típico 1e-4 a 1e-3.
- **Dropout** fuerza redundancia; activo solo en training (Keras escala en inference).
- **MaxNorm** acota la norma de los pesos por unidad.
- **MC Dropout** aproxima incertidumbre bayesiana: mayor std en muestras ambiguas.
- **Stochastic Depth** (con `p` lineal) regulariza redes residuales muy profundas (ResNet, ViT vía DropPath).
- Evitá doble penalización: AdamW(wd=...) **o** `kernel_regularizer` L2, no ambos.

## ✅ Soluciones de los ejercicios

Regularización en Keras 3 sobre un MLP grande en Fashion-MNIST. Las celdas usan la API real de Keras (se validan por AST sin TF). El **Ej. 5** (stochastic depth simulado) es NumPy puro y ejecutable con `assert`.

**Ej. 1 — Sin regularización.** MLP grande `[512,256,128]` que sobreajusta (gap train/val >= 5 pp).

In [ ]:
import numpy as np
from tensorflow import keras

(x_train, y_train), (x_val, y_val) = keras.datasets.fashion_mnist.load_data()
x_train, x_val = x_train / 255., x_val / 255.

def make(reg=None, dropout=0.0):
    layers = [keras.Input((28, 28)), keras.layers.Flatten()]
    for u in (512, 256, 128):
        layers.append(keras.layers.Dense(u, activation="relu", kernel_regularizer=reg))
        if dropout:
            layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(10, activation="softmax"))
    return keras.Sequential(layers)

base = make()
base.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
h = base.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=20, verbose=0)
gap = h.history["accuracy"][-1] - h.history["val_accuracy"][-1]
print(f"gap train-val = {gap*100:.1f} pp (overfitting si >= 5 pp)")

**Ej. 2 — L2.** `kernel_regularizer=l2(1e-3)` en cada Dense: penaliza `sum(w^2)` y suaviza la función.

In [ ]:
from tensorflow import keras

l2 = keras.regularizers.l2(1e-3)
model = make(reg=l2)                       # reutiliza la fabrica del Ej. 1
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
# L2 empuja los pesos hacia 0 -> funcion mas suave -> menor gap train/val.
print("L2(1e-3) en cada Dense; el gap deberia reducirse frente al baseline")

**Ej. 3 — Dropout.** `Dropout(0.3)` entre capas: apaga 30% de activaciones por batch (solo en training).

In [ ]:
from tensorflow import keras

model = make(dropout=0.3)
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
# En training apaga neuronas al azar -> la red no puede co-adaptarse a unas pocas.
# En inferencia dropout se desactiva y las activaciones se escalan automaticamente.
print("Dropout(0.3) entre Dense: reduce co-adaptacion")

**Ej. 4 — MC Dropout.** 100 forwards con `training=True` sobre 1 ejemplo -> `mean ± std` como incertidumbre.

In [ ]:
import numpy as np
from tensorflow import keras

model = make(dropout=0.3)
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
# ... tras entrenar el modelo ...
x = x_val[:1]
probs = np.stack([model(x, training=True).numpy()[0] for _ in range(100)])  # dropout ON
mean, std = probs.mean(0), probs.std(0)
pred = int(mean.argmax())
print(f"clase {pred}: p = {mean[pred]:.3f} +/- {std[pred]:.3f}")
print("std alta -> el modelo esta inseguro en ese ejemplo (incertidumbre bayesiana aprox.)")

**Ej. 5 — Stochastic Depth simulado.** Rate lineal `p_i` en 8 bloques residuales; NumPy puro, ejecutable.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
N = 8

def survival_probs(N, p_max=0.1):
    # bloque i se DROPEA con prob i/(N-1)*p_max; sobrevive con 1 - eso
    return np.array([1 - (i / (N - 1)) * p_max for i in range(N)])

sp = survival_probs(N)
assert sp[0] == 1.0 and abs(sp[-1] - 0.9) < 1e-9   # el 1o siempre vive, el ultimo 90%

def residual_block(x, survive_p, training, rng):
    fx = 0.5 * x                       # "transformacion" F(x) de juguete
    if training:
        if rng.random() < survive_p:
            return x + fx / survive_p  # activo: escala 1/p para mantener la esperanza
        return x                       # dropeado: pasa la identidad
    return x + fx * survive_p          # test: escala por p (como en dropout)

x = np.ones(4)
out = residual_block(x, sp[4], training=True, rng=rng)
assert out.shape == x.shape
print("survival probs:", np.round(sp, 3))
print("los bloques profundos se dropean mas seguido -> red efectiva mas corta")